# IT3212 – Assignment 3 – Task 7: Bagging and Boosting

This notebook builds on **Task 4 (Feature Selection)** and **Tasks 5–6 (Classification and Comparison)**. Task 7 implements two ensemble learning methods using the same 18 selected input features and the existing test/training/validation split.

**Models:** Random Forest (Bagging), AdaBoost with decision stumps (matched control), and AdaBoost with Gaussian Naive Bayes (alternative base-model family). Standalone decision stump and GaussianNB models are included as baselines.

**Target:** `smoking` (`0` = never smoked, `1` = current or former smoker).

**Inputs:** `../data/assignment3/task4_train.csv`, `task4_val.csv`, `task4_test.csv`, and `task4_metadata.json`.

Hyperparameters are selected using validation ROC-AUC. The test set is reserved for final evaluation. This notebook initially reports metrics at the existing threshold of 0.5; operating-threshold selection for the common-recall comparison will be handled separately in the performance-comparison workflow.


A second performance comparison selects model-specific thresholds on validation data to target a common minimum recall, then evaluates those fixed thresholds on the test set. Paired bootstrap intervals quantify uncertainty in precision and recall differences versus Random Forest.

In [1]:
# Imports and configuration
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 18
THRESHOLD = 0.5
DATA_DIR = Path("../data/assignment3")

In [2]:
# Load selected features and data splits
with open(DATA_DIR / "task4_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

FEATURES = metadata["selected_features"]
TARGET = metadata["target"]

train_df = pd.read_csv(DATA_DIR / "task4_train.csv")
val_df = pd.read_csv(DATA_DIR / "task4_val.csv")
test_df = pd.read_csv(DATA_DIR / "task4_test.csv")

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val = val_df[FEATURES], val_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

print(f"Number of features: {len(FEATURES)}")
print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set:       {X_test.shape}")

Number of features: 18
Training set:   (787625, 18)
Validation set: (98453, 18)
Test set:       (98454, 18)


### Bagging: Random Forest

In [3]:
# Tune Random Forest using validation ROC-AUC
rf_candidates = []

rf_tuning_start = time.perf_counter()

for n_estimators in [50, 100]:
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=5,
        min_samples_leaf=50,
        max_features="sqrt",
        bootstrap=True,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    model.fit(X_train, y_train)
    val_prob = model.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_prob)

    rf_candidates.append({
        "model": model,
        "n_estimators": n_estimators,
        "validation_auc": val_auc,
    })

random_forest_tuning_time = time.perf_counter() - rf_tuning_start

best_rf = max(rf_candidates, key=lambda result: result["validation_auc"])
random_forest_model = best_rf["model"]

print("Random Forest – tuning results")
for result in rf_candidates:
    print(
        f'{result["n_estimators"]} trees: '
        f'validation AUC = {result["validation_auc"]:.4f}'
    )

print("\nSelected configuration:")
print(f'Trees: {best_rf["n_estimators"]}')
print(f'Validation AUC: {best_rf["validation_auc"]:.4f}')
print(f'Tuning and training time: {random_forest_tuning_time:.2f} seconds')

Random Forest – tuning results
50 trees: validation AUC = 0.8675
100 trees: validation AUC = 0.8676

Selected configuration:
Trees: 100
Validation AUC: 0.8676
Tuning and training time: 22.28 seconds


### Boosting: AdaBoost with decision stumps (matched control)

In [4]:
# Tune AdaBoost with decision stumps using validation ROC-AUC
ada_candidates = []

ada_tuning_start = time.perf_counter()

for n_estimators in [25, 50]:
    for learning_rate in [0.5, 1.0]:
        model = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(
                max_depth=1,
                random_state=RANDOM_STATE,
            ),
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            random_state=RANDOM_STATE,
        )

        model.fit(X_train, y_train)
        val_prob = model.predict_proba(X_val)[:, 1]
        val_auc = roc_auc_score(y_val, val_prob)

        ada_candidates.append({
            "model": model,
            "n_estimators": n_estimators,
            "learning_rate": learning_rate,
            "validation_auc": val_auc,
        })

ada_tuning_time = time.perf_counter() - ada_tuning_start

best_ada = max(ada_candidates, key=lambda result: result["validation_auc"])
boosting_model = best_ada["model"]

print("AdaBoost – tuning results")
for result in ada_candidates:
    print(
        f'{result["n_estimators"]} estimators, '
        f'learning rate {result["learning_rate"]}: '
        f'validation AUC = {result["validation_auc"]:.4f}'
    )

print("\nSelected configuration:")
print(f'Estimators: {best_ada["n_estimators"]}')
print(f'Learning rate: {best_ada["learning_rate"]}')
print(f'Validation AUC: {best_ada["validation_auc"]:.4f}')
print(f'Tuning and training time: {ada_tuning_time:.2f} seconds')

AdaBoost – tuning results
25 estimators, learning rate 0.5: validation AUC = 0.8645
25 estimators, learning rate 1.0: validation AUC = 0.8699
50 estimators, learning rate 0.5: validation AUC = 0.8699
50 estimators, learning rate 1.0: validation AUC = 0.8729

Selected configuration:
Estimators: 50
Learning rate: 1.0
Validation AUC: 0.8729
Tuning and training time: 92.50 seconds


### Alternative base-model family: Gaussian Naive Bayes

In [5]:
# Fit standalone baselines
stump_start = time.perf_counter()
stump_model = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)
stump_model.fit(X_train, y_train)
stump_training_time = time.perf_counter() - stump_start

gnb_start = time.perf_counter()
gnb_model = GaussianNB()
gnb_model.fit(X_train, y_train)
gnb_training_time = time.perf_counter() - gnb_start

print(f"Standalone decision stump training time: {stump_training_time:.2f} seconds")
print(f"Standalone GaussianNB training time: {gnb_training_time:.2f} seconds")

Standalone decision stump training time: 0.67 seconds
Standalone GaussianNB training time: 0.16 seconds


In [6]:
# Tune AdaBoost with GaussianNB using validation ROC-AUC
gnb_ada_candidates = []
gnb_ada_failures = []
gnb_ada_tuning_start = time.perf_counter()

for n_estimators in [25, 50]:
    for learning_rate in [0.5, 1.0]:
        candidate = AdaBoostClassifier(
            estimator=GaussianNB(),
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            random_state=RANDOM_STATE,
        )

        try:
            candidate.fit(X_train, y_train)
            val_prob = candidate.predict_proba(X_val)[:, 1]
            val_auc = roc_auc_score(y_val, val_prob)
            n_fitted = len(candidate.estimators_)
            final_estimator_error = candidate.estimator_errors_[n_fitted - 1]

            gnb_ada_candidates.append({
                "model": candidate,
                "n_estimators_requested": n_estimators,
                "n_estimators_fitted": n_fitted,
                "learning_rate": learning_rate,
                "validation_auc": val_auc,
                "final_estimator_error": final_estimator_error,
            })
        except (ValueError, FloatingPointError) as error:
            gnb_ada_failures.append({
                "n_estimators": n_estimators,
                "learning_rate": learning_rate,
                "error": str(error),
            })

gnb_ada_tuning_time = time.perf_counter() - gnb_ada_tuning_start

if not gnb_ada_candidates:
    raise RuntimeError(
        "No AdaBoost + GaussianNB configuration trained successfully. "
        "Inspect gnb_ada_failures before proceeding."
    )

best_gnb_ada = max(gnb_ada_candidates, key=lambda result: result["validation_auc"])
gnb_boosting_model = best_gnb_ada["model"]

print("AdaBoost + GaussianNB – tuning results")
for result in gnb_ada_candidates:
    print(
        f'{result["n_estimators_requested"]} requested, '
        f'{result["n_estimators_fitted"]} fitted estimators, '
        f'learning rate {result["learning_rate"]}: '
        f'validation AUC = {result["validation_auc"]:.4f}, '
        f'last estimator error = {result["final_estimator_error"]:.4f}'
    )

if gnb_ada_failures:
    print("\nConfigurations that failed:")
    for failure in gnb_ada_failures:
        print(failure)

print("\nSelected configuration:")
print(f'Requested estimators: {best_gnb_ada["n_estimators_requested"]}')
print(f'Actually fitted estimators: {best_gnb_ada["n_estimators_fitted"]}')
print(f'Learning rate: {best_gnb_ada["learning_rate"]}')
print(f'Validation AUC: {best_gnb_ada["validation_auc"]:.4f}')
print(f'Tuning and training time: {gnb_ada_tuning_time:.2f} seconds')

AdaBoost + GaussianNB – tuning results
25 requested, 25 fitted estimators, learning rate 0.5: validation AUC = 0.8564, last estimator error = 0.4977
25 requested, 10 fitted estimators, learning rate 1.0: validation AUC = 0.8485, last estimator error = 0.5000
50 requested, 30 fitted estimators, learning rate 0.5: validation AUC = 0.8564, last estimator error = 0.4938
50 requested, 10 fitted estimators, learning rate 1.0: validation AUC = 0.8485, last estimator error = 0.5000

Selected configuration:
Requested estimators: 50
Actually fitted estimators: 30
Learning rate: 0.5
Validation AUC: 0.8564
Tuning and training time: 20.23 seconds


### Final evaluation on the test set

In [7]:
# Evaluate fitted models at threshold 0.5
def evaluate_model(model, X, y):
    probabilities = model.predict_proba(X)[:, 1]
    predictions = (probabilities >= THRESHOLD).astype(int)

    return {
        "Accuracy": accuracy_score(y, predictions),
        "Precision": precision_score(y, predictions, zero_division=0),
        "Recall": recall_score(y, predictions, zero_division=0),
        "F1-score": f1_score(y, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y, probabilities),
    }


rf_test_metrics = evaluate_model(random_forest_model, X_test, y_test)
stump_test_metrics = evaluate_model(stump_model, X_test, y_test)
ada_stump_test_metrics = evaluate_model(boosting_model, X_test, y_test)
gnb_test_metrics = evaluate_model(gnb_model, X_test, y_test)
gnb_ada_test_metrics = evaluate_model(gnb_boosting_model, X_test, y_test)

test_results = pd.DataFrame([
    {"Model": "Random Forest", **rf_test_metrics},
    {"Model": "Decision stump", **stump_test_metrics},
    {"Model": "AdaBoost + decision stumps", **ada_stump_test_metrics},
    {"Model": "GaussianNB", **gnb_test_metrics},
    {"Model": "AdaBoost + GaussianNB", **gnb_ada_test_metrics},
])

print(f"Test set results (threshold = {THRESHOLD})")
test_results.round(4)

Test set results (threshold = 0.5)


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Random Forest,0.8092,0.6958,0.9118,0.7893,0.8700
1,Decision stump,0.8097,0.6911,0.9304,0.7931,0.8312
2,AdaBoost + decision stumps,0.8092,0.7049,0.8831,0.7840,0.8748
3,GaussianNB,0.7875,0.6990,0.8042,0.7479,0.8495
4,AdaBoost + GaussianNB,0.8003,0.7171,0.8102,0.7608,0.8575


### Performance comparison at a common recall requirement

In [8]:
# Select thresholds on validation data to meet the target recall
TARGET_RECALL = 0.90

models = {
    "Random Forest": random_forest_model,
    "Decision stump": stump_model,
    "AdaBoost + decision stumps": boosting_model,
    "GaussianNB": gnb_model,
    "AdaBoost + GaussianNB": gnb_boosting_model,
}


def select_threshold_for_recall(y_true, probabilities, target_recall=TARGET_RECALL):
    """Choose a validation threshold maximizing precision subject to recall >= target."""
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)

    eligible = np.flatnonzero(recall[:-1] >= target_recall)
    if len(eligible) == 0:
        raise ValueError(
            f"No threshold achieves the requested validation recall of {target_recall:.3f}."
        )

    eligible_precision = precision[eligible]
    best_precision = eligible_precision.max()

    best_candidates = eligible[eligible_precision == best_precision]
    best_idx = best_candidates[np.argmax(thresholds[best_candidates])]

    return float(thresholds[best_idx]), float(precision[best_idx]), float(recall[best_idx])


validation_thresholds = {}
validation_threshold_rows = []
validation_probabilities = {}

for model_name, model in models.items():
    val_prob = model.predict_proba(X_val)[:, 1]
    validation_probabilities[model_name] = val_prob
    selected_threshold, val_precision, val_recall = select_threshold_for_recall(
        y_val, val_prob, TARGET_RECALL
    )
    validation_thresholds[model_name] = selected_threshold
    validation_threshold_rows.append({
        "Model": model_name,
        "Selected threshold": selected_threshold,
        "Validation precision": val_precision,
        "Validation recall": val_recall,
    })

validation_threshold_results = pd.DataFrame(validation_threshold_rows)
print(f"Thresholds selected on validation data (minimum recall = {TARGET_RECALL:.0%})")
validation_threshold_results.round(4)

Thresholds selected on validation data (minimum recall = 90%)


,Model,Selected threshold,Validation precision,Validation recall
0,Random Forest,0.5247,0.6949,0.9000
1,Decision stump,0.6868,0.6869,0.9310
2,AdaBoost + decision stumps,0.4896,0.6963,0.9002
3,GaussianNB,0.0459,0.6776,0.9000
4,AdaBoost + GaussianNB,0.3059,0.6901,0.9001


### Test-set evaluation at validation-selected thresholds

In [9]:
def evaluate_at_threshold(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)

    tn = np.sum((y_true == 0) & (predictions == 0))
    fp = np.sum((y_true == 0) & (predictions == 1))
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "Specificity": specificity,
        "F1-score": f1_score(y_true, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
    }


test_probabilities = {}
common_recall_rows = []

for model_name, model in models.items():
    test_prob = model.predict_proba(X_test)[:, 1]
    test_probabilities[model_name] = test_prob
    common_recall_rows.append({
        "Model": model_name,
        **evaluate_at_threshold(
            y_test,
            test_prob,
            validation_thresholds[model_name],
        ),
    })

common_recall_test_results = pd.DataFrame(common_recall_rows)
print(f"Test-set results using thresholds selected for {TARGET_RECALL:.0%} validation recall")
common_recall_test_results.round(4)

Test-set results using thresholds selected for 90% validation recall


,Model,Threshold,Accuracy,Precision,Recall,Specificity,F1-score,ROC-AUC
0,Random Forest,0.5247,0.8078,0.6981,0.8981,0.7496,0.7855,0.8700
1,Decision stump,0.6868,0.8097,0.6911,0.9304,0.7319,0.7931,0.8312
2,AdaBoost + decision stumps,0.4896,0.8095,0.7002,0.8991,0.7518,0.7873,0.8748
3,GaussianNB,0.0459,0.7959,0.6819,0.8984,0.7298,0.7753,0.8495
4,AdaBoost + GaussianNB,0.3059,0.8047,0.6938,0.8981,0.7444,0.7828,0.8575


### Paired bootstrap uncertainty analysis

In [10]:
# Paired bootstrap confidence intervals versus Random Forest
N_BOOTSTRAPS = 2000
BOOTSTRAP_RANDOM_STATE = 18
REFERENCE_MODEL = "Random Forest"

y_test_array = np.asarray(y_test)
test_predictions_common_recall = {
    name: (prob >= validation_thresholds[name]).astype(int)
    for name, prob in test_probabilities.items()
}

rng = np.random.default_rng(BOOTSTRAP_RANDOM_STATE)
model_names = list(models.keys())
bootstrap_differences = {
    name: {"precision": [], "recall": []}
    for name in model_names if name != REFERENCE_MODEL
}

n_test = len(y_test_array)
for _ in range(N_BOOTSTRAPS):
    sample_idx = rng.integers(0, n_test, size=n_test)
    y_sample = y_test_array[sample_idx]

    ref_pred = test_predictions_common_recall[REFERENCE_MODEL][sample_idx]
    ref_tp = np.sum((y_sample == 1) & (ref_pred == 1))
    ref_fp = np.sum((y_sample == 0) & (ref_pred == 1))
    ref_fn = np.sum((y_sample == 1) & (ref_pred == 0))
    ref_precision = ref_tp / (ref_tp + ref_fp) if (ref_tp + ref_fp) else 0.0
    ref_recall = ref_tp / (ref_tp + ref_fn) if (ref_tp + ref_fn) else 0.0

    for name in bootstrap_differences:
        pred = test_predictions_common_recall[name][sample_idx]
        tp = np.sum((y_sample == 1) & (pred == 1))
        fp = np.sum((y_sample == 0) & (pred == 1))
        fn = np.sum((y_sample == 1) & (pred == 0))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0

        bootstrap_differences[name]["precision"].append(precision - ref_precision)
        bootstrap_differences[name]["recall"].append(recall - ref_recall)

bootstrap_rows = []
for name, metric_samples in bootstrap_differences.items():
    precision_low, precision_high = np.percentile(metric_samples["precision"], [2.5, 97.5])
    recall_low, recall_high = np.percentile(metric_samples["recall"], [2.5, 97.5])
    bootstrap_rows.append({
        "Comparison": f"{name} minus {REFERENCE_MODEL}",
        "Precision difference": (
            common_recall_test_results.loc[
                common_recall_test_results["Model"] == name, "Precision"
            ].iloc[0]
            - common_recall_test_results.loc[
                common_recall_test_results["Model"] == REFERENCE_MODEL, "Precision"
            ].iloc[0]
        ),
        "Precision difference 95% CI": f"[{precision_low:.4f}, {precision_high:.4f}]",
        "Recall difference": (
            common_recall_test_results.loc[
                common_recall_test_results["Model"] == name, "Recall"
            ].iloc[0]
            - common_recall_test_results.loc[
                common_recall_test_results["Model"] == REFERENCE_MODEL, "Recall"
            ].iloc[0]
        ),
        "Recall difference 95% CI": f"[{recall_low:.4f}, {recall_high:.4f}]",
    })

paired_bootstrap_results = pd.DataFrame(bootstrap_rows)
print(f"Paired bootstrap results ({N_BOOTSTRAPS} resamples; reference = {REFERENCE_MODEL})")
paired_bootstrap_results

Paired bootstrap results (2000 resamples; reference = Random Forest)


,Comparison,Precision difference,Precision difference 95% CI,Recall difference,Recall difference 95% CI
0,Decision stump minus Random Forest,-0.006970,"[-0.0079, -0.0060]",0.032363,"[0.0306, 0.0341]"
1,AdaBoost + decision stumps minus Random Forest,0.002131,"[0.0011, 0.0032]",0.001036,"[-0.0007, 0.0027]"
2,GaussianNB minus Random Forest,-0.016185,"[-0.0176, -0.0149]",0.000363,"[-0.0018, 0.0023]"
3,AdaBoost + GaussianNB minus Random Forest,-0.004298,"[-0.0053, -0.0033]",0.000026,"[-0.0018, 0.0017]"


### Model size and training time

In [11]:
# Compare training time and model complexity
rf_depths = [tree.get_depth() for tree in random_forest_model.estimators_]
stump_depth = stump_model.get_depth()
ada_stump_depths = [tree.get_depth() for tree in boosting_model.estimators_]

complexity_results = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Training time (s)": random_forest_tuning_time,
        "Number of fitted estimators": len(random_forest_model.estimators_),
        "Maximum tree depth": max(rf_depths),
    },
    {
        "Model": "Decision stump",
        "Training time (s)": stump_training_time,
        "Number of fitted estimators": 1,
        "Maximum tree depth": stump_depth,
    },
    {
        "Model": "AdaBoost + decision stumps",
        "Training time (s)": ada_tuning_time,
        "Number of fitted estimators": len(boosting_model.estimators_),
        "Maximum tree depth": max(ada_stump_depths),
    },
    {
        "Model": "GaussianNB",
        "Training time (s)": gnb_training_time,
        "Number of fitted estimators": 1,
        "Maximum tree depth": "N/A",
    },
    {
        "Model": "AdaBoost + GaussianNB",
        "Training time (s)": gnb_ada_tuning_time,
        "Number of fitted estimators": len(gnb_boosting_model.estimators_),
        "Maximum tree depth": "N/A",
    },
])

complexity_results

,Model,Training time (s),Number of fitted estimators,Maximum tree depth
0,Random Forest,22.277856,100,5
1,Decision stump,0.674122,1,1
2,AdaBoost + decision stumps,92.498032,50,1
3,GaussianNB,0.155409,1,N/A
4,AdaBoost + GaussianNB,20.231027,30,N/A


### AdaBoost estimator inspection

In [12]:
print("AdaBoost + decision stumps: first 10 estimator errors")
print(boosting_model.estimator_errors_[:10].round(4))
print("First 10 estimator weights:")
print(boosting_model.estimator_weights_[:10].round(4))

print("\nAdaBoost + GaussianNB: first 10 estimator errors")
print(gnb_boosting_model.estimator_errors_[:10].round(4))
print("First 10 estimator weights:")
print(gnb_boosting_model.estimator_weights_[:10].round(4))
print(f"\nGaussianNB ensemble fitted {len(gnb_boosting_model.estimators_)} estimators.")


AdaBoost + decision stumps: first 10 estimator errors
[0.1937 0.297  0.4041 0.4452 0.467  0.4688 0.4771 0.4768 0.468  0.4722]
First 10 estimator weights:
[1.4259 0.8616 0.3884 0.2202 0.1321 0.1249 0.0917 0.0929 0.1282 0.1112]

AdaBoost + GaussianNB: first 10 estimator errors
[0.2156 0.3358 0.3922 0.4192 0.4247 0.3937 0.375  0.419  0.4168 0.4384]
First 10 estimator weights:
[0.6458 0.3411 0.219  0.1631 0.1518 0.2159 0.2554 0.1633 0.1679 0.1239]

GaussianNB ensemble fitted 30 estimators.
